In [1]:
!pip install vllm -q

In [10]:
import requests
import subprocess
import signal
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HUGGINGFACE_HUB_TOKEN'))



In [11]:
def list_model_ids():
  url = "http://localhost:8000/v1/models"
  response = requests.get(url)
  return [model['id'] for model in response.json()['data']]

In [12]:
def stop_vllm_server():
    """
    Stop a background vLLM server cleanly (SIGTERM).
    Falls back to SIGKILL if needed.
    """
    try:
        # Find vLLM server PIDs
        result = subprocess.check_output(
            ["pgrep", "-f", "vllm serve"],
            text=True
        ).strip()

        if not result:
            print("No vLLM server running.")
            return

        pids = result.splitlines()
        for pid in pids:
            print(f"Stopping vLLM server (PID {pid})...")
            subprocess.run(["kill", pid])

    except subprocess.CalledProcessError:
        print("No vLLM server running.")

In [13]:
def get_response(system_prompt, prompt, model, temperature=0.01, max_tokens=150):

  url = "http://localhost:8000/v1/chat/completions"

  payload = {
      "model": model,
      "messages": [
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": prompt}
      ],
      "temperature": temperature,
      "max_tokens": max_tokens,
  }

  response = requests.post(url, json=payload)
  return response.json()['choices'][0]['message']['content']


In [15]:
!nohup vllm serve meta-llama/Llama-3.2-1B-Instruct \
  --enable-lora \
  --lora-modules tuned_model=moo3030/Llama-3.2-1B-QLoRA-Summarizer-adapters \
  --gpu-memory-utilization 0.7 \
  --max-model-len 2048 \
  --max-num-seqs 4 \
  > vllm.log 2>&1 &


In [7]:
while True:
  try:
    list_model_ids()
    break
  except:
    pass

system_prompt  = "You are a helpful assistant that summarizes conversations."
prompt = "Victoria: God I'm really broke, I spent way to much this month \ud83d\ude2b\nVictoria: At least we get paid soon..\nMagda: Yeah, don't remind me, I know the feeling\nMagda: I just paid my car insurance, I feel robbed \ud83d\ude02\nVictoria: Thankfully mine is paid for the rest of the year \ud83d\ude4f\nMagda: \ud83d\udc4c"

model_ids = list_model_ids()
base_model_id = model_ids[0]
lora_model_id = model_ids[1]

base_response = get_response(system_prompt, prompt, base_model_id)
lora_response = get_response(system_prompt, prompt, lora_model_id)

In [8]:
base_response

'Here\'s a summary of the conversation:\n\nVictoria and Magda are discussing their financial situation. Victoria mentions that she\'s broke and spent too much this month, while Magda shares her own experience of being broke. They both express relief that they get paid soon, which will help alleviate their financial stress. Magda jokes about being "robbed" after paying her car insurance, but ultimately agrees with Victoria that their finances are improving.'

In [9]:
lora_response

'Victoria and Magda are broke. They get paid soon.'

In [10]:
import requests
import json

def stream_response(system_prompt, prompt, model, temperature=0.01, max_tokens=150):
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": True,
    }

    with requests.post(url, json=payload, stream=True) as response:
        response.raise_for_status()

        for line in response.iter_lines():
            if not line:
                continue

            # vLLM uses SSE-style "data: ..."
            decoded = line.decode("utf-8")
            if decoded.startswith("data: "):
                data = decoded[len("data: "):]

                if data == "[DONE]":
                    break

                chunk = json.loads(data)
                delta = chunk["choices"][0]["delta"]

                if "content" in delta:
                    yield delta["content"]


In [11]:
for token in stream_response(
    system_prompt="You are a helpful assistant.",
    prompt="Explain dynamic batching in simple terms.",
    model="meta-llama/Llama-3.2-1B-Instruct",
):
    print(token, end="", flush=True)


Dynamic batching is a technique used in programming to group multiple operations together and execute them in a single pass, reducing the overhead of multiple function calls.

Imagine you're making multiple requests to a server, like checking your email, updating your social media, and sending a message. If you had to make each request individually, you'd make 3 separate API calls, which would be slower and more resource-intensive.

Dynamic batching works in a similar way. Instead of making each request individually, you group multiple operations together and execute them in a single pass. This reduces the overhead of multiple function calls and makes the process faster.

Here's an example:

```python
def dynamic_batching():
    # Group multiple operations together
    update_email()
    send_message

In [12]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

start_time = time.time()

def fire_requests_concurrently(
    system_prompt,
    prompt,
    model,
    temperature=0.01,
    max_tokens=150,
    n_requests=10
):
    """
    Fire multiple requests at the same time and return all responses.
    """
    responses = []

    with ThreadPoolExecutor(max_workers=n_requests) as executor:
        futures = [
            executor.submit(
                get_response,
                system_prompt,
                prompt,
                model,
                temperature,
                max_tokens,
            )
            for _ in range(n_requests)
        ]

        for future in as_completed(futures):
            responses.append(future.result())

    return responses


outputs = fire_requests_concurrently(
    system_prompt="You are a helpful assistant.",
    prompt="Summarize the benefits of dynamic batching in vLLM.",
    model="meta-llama/Llama-3.2-1B-Instruct",
    n_requests=40,
    temperature=1
)



for i, out in enumerate(outputs, 1):
    print(f"Response {i}:\n{out}\n")

end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken: {total_time} seconds")


Response 1:
Involuntary Learning (IL) or Dynamic Batching in Very Large Learning Models (VLLMs) offers several benefits, including:

1. **Increased Perceptual Consistency**: Dynamic batching ensures that the model processes the entire input sequence in a synchronized manner, improving the overall perceptual consistency and reducing inconsistencies in the learned representations.

2. **Reduced Overfitting**: By dynamically batching similar components together, the model's capacity to adapt to the input sequence is maximized, reducing overfitting and improving its ability to generalize to new data.

3. **Improved Model Efficiency**: Dynamic batching helps the model allocate its computational resources more efficiently, as the most computationally expensive components are grouped together, reducing unnecessary processing.

4. **Enhanced

Response 2:
In VLLM (Very Long Lexical Material), dynamic batching is a technique used to optimize the processing of large, serialized lexicon elements, 

In [14]:
stop_vllm_server()

No vLLM server running.


In [17]:
!vllm bench serve \
  --backend vllm \
  --model tuned_model \
  --tokenizer meta-llama/Llama-3.2-1B-Instruct \
  --dataset-name random \
  --random-input-len 512 \
  --random-output-len 256 \
  --num-prompts 20 \
  --max-concurrency 4

2025-12-31 15:47:37.631320: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-31 15:47:37.648841: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767196057.669903   14015 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767196057.676361   14015 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767196057.692555   14015 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [14]:
stop_vllm_server()

No vLLM server running.
